Small experiment showing that using the problem of OOD for hyperparameter finding is problematic in that it is unreliable. For some dataset model combination it works depending on the metric used. For other it may find a good solution but the solution might also be randomly worse. Generally this shows that a combination good for OOD detection can be suboptimal for the task of caonicalization.

In [ ]:
import gc
import torch
import numpy as np
from src.utils.sampling import BatchNegativeSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#look for experiment files in parents
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "bigger_emnist"

default_architecutre_mapping = {
    "mnist":"resnet_small",
    "bigger_mnist":"resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist":"bigger_extended_resnet_small",
    "coil100":"coil_resnet_small",
    "tu_berlin":"bi_lstm",
    "modelnet10":"pointnetplus",
}



architecture = default_architecutre_mapping[dataset]
budget = 60

In [ ]:
#NOTE already rerun for using the whole embedding cache

In [ ]:
from data.get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)
dataset_dict = get_dataset(dataset_info,path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:


dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
from src.utils.eval.vis import vis_dataset, plt_setup_paper

batch_size = next(iter(train_loader))[0].shape[0]
vis_dataset(train_loader,val_loader,test_loader_transformed)

In [ ]:
from model.train import train_and_get_model,train_or_load_energy_model
from model.get_model import get_network
from src.utils.eval.main_model import evaluate_base_model

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture, "uncertainty")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path,transform_name, f"{safe}.json")

In [ ]:
model = get_network(dataset_info,architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train= f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model,model_dir_path,modelname, train_loader, val_loader , trainer_kwargs= {
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs,
        "precision": "16-mixed",
},load_if_exists=True)



In [ ]:
model.eval().to(device)
res = evaluate_base_model(model, test_loader_transformed, device)
print(res)
res = evaluate_base_model(model, test_loader, device)
print(res)

In [ ]:
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]


In [ ]:
from src.utils.transforms.apply import grid_resample
from data.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
                name=dataset_info.transform_seq_name,
                resample_method=dataset_info.resample_method,
    init_method="sobol"
    ).to(device)

In [ ]:
from embedding_cache import LayerEmbeddingCache



cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"
import embedding_cache
import importlib
importlib.reload(embedding_cache)



transform_name = dataset_info.transform_seq_name

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"
from data.get_dataset import get_layer_embedding_cache_config,create_layer_embedding_cache
cache_config = get_layer_embedding_cache_config(dataset, architecture,transform_name=None,dataset_info=dataset_info)

In [ ]:
cache_config

In [ ]:
train_cache =create_layer_embedding_cache(model, train_loader_no_shuffle,cache_config, embedding_cache_path, device=device)


In [ ]:
from search.random_search import RSLR
random_search  = RSLR(initial_samples=60,local_runs=1,local_max_steps=0)

In [ ]:

model.cuda().eval()

In [ ]:
import hyper_param.ood.base_prepare
hyper_param.ood.base_prepare.OOD_PARAM_SAMPLERS
print("")

In [ ]:
from src.utils.sampling_strategy import TransformLatentSamplingStrategy

sampling_strategy = TransformLatentSamplingStrategy(transform_seq,clip_data=True)
num_negatives =60
sampler =BatchNegativeSampler(sampling_strategy,number_of_negatives=num_negatives).to(device)


In [ ]:
id_x_mode2, id_y_mode2 = [], []
ood_x_mode2, ood_y_mode2 = [], []


torch.manual_seed(42)

with torch.no_grad():
    for batch in val_loader:
        x_orig, y_orig = batch
        batch_size = x_orig.size(0)
        M = 1 + num_negatives  # number of candidates per sample

        # Sample positives + negatives (DO NOT change sampler)
        modified_batch = sampler((x_orig.to(device), y_orig.to(device)))
        x_all = modified_batch[0].to(device)   # shape: (batch_size * M, C, H, W)
        y_orig = y_orig.to(device)             # shape: (batch_size,)

        # Forward pass over flattened candidates (batched for memory efficiency)
        for i in range(0, x_all.shape[0], batch_size):
            x_batch = x_all[i:i+batch_size]
            logits_batch = model(x_batch)
            if i == 0:
                logits_all = logits_batch
            else:
                logits_all = torch.cat((logits_all, logits_batch), dim=0)

        # Group back into per-sample candidates: (B, M, num_classes)
        logits_all = torch.stack(logits_all.chunk(M, dim=0), dim=1)
        x_all = torch.stack(x_all.chunk(M, dim=0), dim=1)

        # --- ID Selection: candidate with highest logit for the true class ---
        y_idx = y_orig.view(-1, 1, 1).expand(-1, M, 1)
        true_class_logits = logits_all.gather(2, y_idx).squeeze(2)
        id_indices = torch.argmax(true_class_logits, dim=1)  # (B,)

        # --- OOD Selection: randomly pick another candidate (not the ID one) ---
        all_indices = torch.arange(M, device=device).unsqueeze(0).repeat(batch_size, 1)
        rand_indices = torch.randint(0, M - 1, (batch_size,), device=device)
        # shift indices to avoid id_indices
        ood_indices = torch.where(rand_indices >= id_indices, rand_indices + 1, rand_indices)

        # --- Gather paired samples ---
        ar = torch.arange(batch_size, device=device)
        id_x_batch = x_all[ar, id_indices]
        ood_x_batch = x_all[ar, ood_indices]

        # Append results
        id_x_mode2.append(id_x_batch.cpu())
        id_y_mode2.append(y_orig.cpu())
        ood_x_mode2.append(ood_x_batch.cpu())
        ood_y_mode2.append(y_orig.cpu())

# Concatenate results from all batches
id_x_mode2 = torch.cat(id_x_mode2, dim=0)
id_y_mode2 = torch.cat(id_y_mode2, dim=0)
ood_x_mode2 = torch.cat(ood_x_mode2, dim=0)
ood_y_mode2 = torch.cat(ood_y_mode2, dim=0)

# Build datasets and loaders
dataset_id_mode2 = torch.utils.data.TensorDataset(id_x_mode2, id_y_mode2)
dataset_ood_mode2 = torch.utils.data.TensorDataset(ood_x_mode2, ood_y_mode2)
loader_id_mode2 = torch.utils.data.DataLoader(dataset_id_mode2, batch_size=dataset_info.batch_size, shuffle=False)
loader_ood_mode2 = torch.utils.data.DataLoader(dataset_ood_mode2, batch_size=dataset_info.batch_size, shuffle=False)


In [ ]:
#plot an example from the dataset to see wether they show the same image
#check if data is iamge data
if is_image_data and id_x_mode2.shape[1] in [1,]:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8, 4))
    for i in range(5):
        plt.subplot(2, 5, i + 1)
        plt.imshow(id_x_mode2[i].cpu().squeeze(), cmap='gray')
        plt.title(f"ID Mode 1 - {id_y_mode2[i].item()}")
        plt.axis('off')
        plt.subplot(2, 5, i + 6)
        plt.imshow(ood_x_mode2[i].cpu().squeeze(), cmap='gray')
        plt.title(f"OOD Mode 1 - {id_y_mode2[i].item()}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
from torch.utils.data import DataLoader, Subset

val_dataset = val_loader_transformed.dataset  # assume indexable dataset
n_samples = len(val_dataset)
rng = np.random.default_rng(seed=42)
shuffled_indices = rng.permutation(n_samples)
val_dataset_preshuffled = Subset(val_dataset, shuffled_indices)

val_loader_transformed_preshuffled = DataLoader(
    val_dataset_preshuffled,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers,
)

In [ ]:
n_samples_fraction = int(1/6 * len(val_dataset))
shuffled_indices = rng.permutation(n_samples)[:n_samples_fraction]
val_dataset_preshuffled_small = Subset(val_dataset, shuffled_indices)
val_loader_transformed_preshuffled_small = DataLoader(
    val_dataset_preshuffled_small,
    batch_size=dataset_info.batch_size,
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=val_loader_transformed.persistent_workers,
)


In [ ]:
import os
import json
from datetime import datetime
import optuna
import numpy as np
import pandas as pd

from hyper_param.ood.base_prepare import (
    run_ood_study,
    get_default_ood_params,
    get_best_ood_params_from_study,
    create_ood_problem,
)
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search
from search.random_search import RSLR

# configure
detector = "knn_mixed"
detector2 = "knn_trap"
ood_objectives = ["auroc", "paired_ood_acc", "fpr95"]
n_trials_ood = 100
n_trials_search_small = 60
n_trials_search_large = 30
num_runs = 6

# Search optimizers
optimizer_search_eval = RSLR(initial_samples=60, local_runs=1, local_max_steps=0) # For final evaluation
optimizer_search_small = RSLR(initial_samples=10, local_runs=1, local_max_steps=0)
optimizer_search_large = RSLR(initial_samples=60, local_runs=1, local_max_steps=0)




# storage dir for metadata
ood_studies_dir = os.path.join(
    current_path,
    "experiment_files",
    "experiment_hyperparameter_opt_ood",
    dataset,
    architecture,
    getattr(dataset_info, "transform_seq_name", "default"),
)
os.makedirs(ood_studies_dir, exist_ok=True)

transform_seq_arg = transform_seq

all_results = []

@torch.no_grad()
def run_experiment(
    run_idx,
    exp_name,
    id_loader,
    ood_loader,
    objectives,
    n_trials,
    val_loader_for_search,
    optimizer_for_study=optimizer_search_large,  # do NOT delete this outside object
):
    """Helper to run one full experiment iteration. Memory-leak-aware version."""
    print(f"\n===== Running Experiment: {exp_name} (Run {run_idx + 1}/{num_runs}) =====")
    gc.collect()
    torch.cuda.empty_cache()

    run_dir = os.path.join(ood_studies_dir, exp_name, f"run_{run_idx}")
    os.makedirs(run_dir, exist_ok=True)

    run_results = []

    for objective in objectives:
        train_cache = create_layer_embedding_cache(
            model, train_loader_no_shuffle, cache_config, embedding_cache_path, device=device
        )
        print(f"== Processing objective={objective} ==")
        gc.collect()
        torch.cuda.empty_cache()

        eval_results_path = os.path.join(run_dir, f"{objective}_eval.json")
        params_path = os.path.join(run_dir, f"{objective}_params.json")

        # If both cached files exist, load and continue (no heavy objects created)
        if os.path.exists(eval_results_path) and os.path.exists(params_path):
            print(f"  Found cached results for objective '{objective}'. Loading...")
            with open(eval_results_path, 'r') as f:
                metrics = json.load(f)
            with open(params_path, 'r') as f:
                best_params = json.load(f)

            search_acc = float(metrics["accuracy_mean"])
            search_acc_std = float(metrics["accuracy_std"])
            search_acc_se = float(metrics["accuracy_se"])
            print(f"  Loaded Search Accuracy: {search_acc:.4f} (+/- {search_acc_std:.4f})")

        else:
            # --- Optimization Step ---
            print(f"  Optimizing {detector} for objective={objective}...")
            study_name = f"ood_{detector}_{objective}_{exp_name}_run{run_idx}"
            storage_path = None  # No DB persistence

            is_direct_search = objective == "search"

            # Build objective kwargs (keep references minimal)
            if is_direct_search:
                report_fraction = 0.1
                if val_loader_for_search == val_loader_transformed_preshuffled_small:
                    report_fraction = 0.2
                    #for coil100 we can use a larger fraction
                    if dataset == "coil100":
                        report_fraction = 0.3
                    if dataset == "tu_berlin":
                        report_fraction = 0.4

                # NOTE: we still pass model/train_cache etc. If run_ood_study stores them internally,
                # it could retain references — so we rely on run_ood_study to not leak. After
                # study completes we will explicitly clear local references.
                objective_kwargs = {
                    "optimizer": optimizer_for_study,
                    "model": model,
                    "train_cache": train_cache,
                    "val_loader": val_loader_for_search,
                    "transform_seq": transform_seq_arg,
                    "dataset_info": dataset_info,
                    "architecture": architecture,
                    "device": str(device),
                    "report_fraction": report_fraction,
                    "repeats": 1,
                }
            else:
                objective_kwargs = {
                    "model": model,
                    "train_cache": train_cache,
                    "id_loader": id_loader,
                    "ood_loader": ood_loader,
                    "transform_seq": transform_seq_arg,
                    "dataset_info": dataset_info,
                    "architecture": architecture,
                    "device": str(device),
                    "metric": objective,
                    "check_percent": 0.1,
                    "prune_at": 0.1,
                    "max_batches": None,
                    "show_progress": False,
                }

            study = run_ood_study(
                study_name=study_name,
                storage_path=storage_path,
                detector_name=detector,
                objective_type=objective,
                objective_kwargs=objective_kwargs,
                n_trials=n_trials,
            )


            # get best params or defaults
            if study is None:
                best_params = get_default_ood_params(detector)
                print(f"  Detector {detector} is parameterless or study skipped; using defaults.")
            else:
                best_params = get_best_ood_params_from_study(study)
                try:
                    print(f"  Best value for {objective}: {study.best_value}")
                except Exception:
                    pass

            # Save best params (convert any numpy/np types to python builtins if necessary)
            with open(params_path, 'w') as f:
                json.dump(json.loads(json.dumps(best_params, default=lambda x: x.tolist() if hasattr(x, "tolist") else str(x))), f, indent=2)
            print(f"  Saved best params to {params_path}")

            # --- Evaluation Step ---
            print(f"  Evaluating final {detector} with best params on search task...")

            # create problem (may keep references to model etc.)
            problem = create_ood_problem(
                detector_name=detector,
                params=best_params,
                model=model,
                train_cache=train_cache,
                transform_seq=transform_seq_arg,
                dataset_info=dataset_info,
                architecture=architecture,
                device=str(device),
            )



            metrics = load_or_run_evaluate_confidence_and_search(
                model=model,
                optimizer=optimizer_search_eval,
                problem=problem,
                test_loader=test_loader_transformed,
                save_path=eval_results_path,
                max_batch_override=dataset_info.batch_size_search,
                show_progress=True,
                repeats=3,
                return_per_run=True,  # Get aggregated metrics
                overwrite=True,  # Overwrite eval, not opt study
                store_val=False,
            )

            # Pull out results, convert to python floats
            search_acc = float(metrics["accuracy_mean"])
            search_acc_std = float(metrics["accuracy_std"])
            search_acc_se = float(metrics["accuracy_se"])
            print(f"  Search Accuracy: {search_acc:.4f} (+/- {search_acc_std:.4f})")

            # print current cuda memory usage
            try:
                print(f"  CUDA Memory Allocated: {torch.cuda.memory_allocated() / (1024 ** 3):.2f} GB")
            except Exception:
                pass


            study = None
            problem = None
            optimizer_eval = None
            metrics = None
            best_params = None
            gc.collect()
            torch.cuda.empty_cache()

            # After evaluation we saved eval json and params json - reload lightweight dicts
            if os.path.exists(eval_results_path) and os.path.exists(params_path):
                with open(eval_results_path, 'r') as f:
                    metrics = json.load(f)
                with open(params_path, 'r') as f:
                    best_params = json.load(f)
                search_acc = float(metrics["accuracy_mean"])
                search_acc_std = float(metrics["accuracy_std"])
                search_acc_se = float(metrics["accuracy_se"])
                print(f"  Loaded Search Accuracy: {search_acc:.4f} (+/- {search_acc_std:.4f})")
            else:
                # Fallback if something unexpected happened
                print("  Warning: evaluation files not found after evaluation. Setting metrics to 0.")
                search_acc = 0.0
                search_acc_std = 0.0
                best_params = get_default_ood_params(detector)

        # Final per-objective append (keep best_params small or already json-serializable)
        run_results.append({
            "exp_name": exp_name,
            "run_idx": run_idx,
            "objective": objective,
            "search_accuracy": float(search_acc),
            "search_accuracy_std": float(search_acc_std),
            "search_accuracy_se": float(search_acc_se),
            "best_params": best_params,
        })

        # Clear heavy locals for the next objective
        best_params = None
        metrics = None
        gc.collect()
        torch.cuda.empty_cache()

    return run_results


# --- Experiment 1: OOD tuning on val vs val_transformed ---
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="val_vs_val_transformed",
        id_loader=val_loader,
        ood_loader=val_loader_transformed,
        objectives=ood_objectives,
        n_trials=n_trials_ood,
        val_loader_for_search=val_loader_transformed_preshuffled,
    )
    all_results.extend(results)




# --- Experiment 2: OOD tuning on mode2 datasets ---
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="mode2_id_vs_ood",
        id_loader=loader_id_mode2,
        ood_loader=loader_ood_mode2,
        objectives=ood_objectives,
        n_trials=n_trials_ood,
        val_loader_for_search=val_loader_transformed_preshuffled,
    )
    all_results.extend(results)
# --- Experiment 3: Direct search optimization (small) ---
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="direct_search_small",
        id_loader=None,
        ood_loader=None,
        objectives=["search"],
        n_trials=n_trials_search_small,
        val_loader_for_search=val_loader_transformed_preshuffled,
        optimizer_for_study=optimizer_search_small,
    )
    all_results.extend(results)
# --- Experiment 4: Direct search optimization (large) ---
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="direct_search_large",
        id_loader=None,
        ood_loader=None,
        objectives=["search"],
        n_trials=n_trials_search_large,
        val_loader_for_search=val_loader_transformed_preshuffled,
    )
    all_results.extend(results)


#restricted to small val set to save time
for i in range(num_runs):
    results = run_experiment(
        run_idx=i,
        exp_name="direct_search_val_restricted",
        id_loader=None,
        ood_loader=None,
        objectives=["search"],
        n_trials=n_trials_search_small,
        val_loader_for_search=val_loader_transformed_preshuffled_small,
    )
    all_results.extend(results)

# --- Experiment 6: Default parameters (no optimization, no objective) ---
for i in range(num_runs):
    print(f"\n===== Running Experiment: default_params (Run {i+1}/{num_runs}) =====")
    gc.collect()
    torch.cuda.empty_cache()

    run_dir = os.path.join(ood_studies_dir, "default_params", f"run_{i}")
    os.makedirs(run_dir, exist_ok=True)

    eval_results_path = os.path.join(run_dir, "default_eval.json")
    params_path = os.path.join(run_dir, "default_params.json")

    if os.path.exists(eval_results_path) and os.path.exists(params_path):
        print("  Found cached default results. Loading...")
        with open(eval_results_path, 'r') as f:
            metrics = json.load(f)
        with open(params_path, 'r') as f:
            best_params = json.load(f)
        search_acc = metrics["accuracy_mean"]
        search_acc_std = metrics["accuracy_std"]
        search_acc_se = metrics["accuracy_se"]

    else:
        # --- Use default parameters ---
        best_params = get_default_ood_params(detector)
        with open(params_path, 'w') as f:
            json.dump(best_params, f, indent=2)
        print(f"  Using default params: {best_params}")

        # --- Evaluation Step ---
        problem = create_ood_problem(
            detector_name=detector,
            params=best_params,
            model=model,
            train_cache=train_cache,
            transform_seq=transform_seq_arg,
            dataset_info=dataset_info,
            architecture=architecture,
            device=str(device),
        )

        metrics = load_or_run_evaluate_confidence_and_search(
            model=model,
            optimizer=optimizer_search_eval,
            problem=problem,
            test_loader=test_loader_transformed,
            save_path=eval_results_path,
            max_batch_override=dataset_info.batch_size_search,
            show_progress=True,
            repeats=3,
            return_per_run=True,
            overwrite=True,
            store_val=False,
        )
        search_acc = metrics["accuracy_mean"]
        search_acc_std = metrics["accuracy_std"]
        search_acc_se = metrics["accuracy_se"]

    all_results.append({
        "exp_name": "default_params",
        "run_idx": i,
        "objective": "none",   # mark explicitly that there was no optimization
        "search_accuracy": search_acc,
        "search_accuracy_std": search_acc_std,
        "search_accuracy_se": search_acc_se,
        "best_params": best_params,
    })




print("\n\n" + "="*20 + " FINAL RESULTS SUMMARY " + "="*20)
results_df = pd.DataFrame(all_results)

# Primary metric: mean and std of the mean accuracies across optimization runs
# This captures the variability from the optimization process itself
summary = results_df.groupby(['exp_name', 'objective']).agg({
    'search_accuracy': ['mean', 'std', 'max'],
    'search_accuracy_std': 'mean'  # Also report average evaluation uncertainty
}).reset_index()

# Flatten column names
summary.columns = ['exp_name', 'objective', 'mean_accuracy', 'std_across_runs', 'max_accuracy', 'mean_eval_std']

# Keep numeric version for plotting
summary_numeric = summary.copy()

# Format for display
summary_display = summary.copy()
summary_display['mean_accuracy'] = summary_display['mean_accuracy'].apply(lambda x: f"{x:.4f}")
summary_display['std_across_runs'] = summary_display['std_across_runs'].apply(lambda x: f"{x:.4f}")
summary_display['max_accuracy'] = summary_display['max_accuracy'].apply(lambda x: f"{x:.4f}")
summary_display['mean_eval_std'] = summary_display['mean_eval_std'].apply(lambda x: f"{x:.4f}")

print("\nSummary Statistics:")
print("- mean_accuracy: Average search accuracy across optimization runs")
print("- std_across_runs: Std dev across optimization runs (optimization variability)")
print("- max_accuracy: Best accuracy achieved across all optimization runs")
print("- mean_eval_std: Average evaluation uncertainty within each run")
print()
print(summary_display.to_string(index=False))

# Save detailed results
results_path = os.path.join(ood_studies_dir, "final_experiment_results.json")
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\nDetailed results saved to: {results_path}")

# Save summary table
summary_path = os.path.join(ood_studies_dir, "results_summary.csv")
results_df.to_csv(summary_path, index=False)
print(f"Summary table saved to: {summary_path}")


In [ ]:
results_df

In [ ]:
#rename experiments val_vs_val_transformed becomes val_vs_transformed
#mode2_id_vs_ood becomes MSP_vs_transformed
#direct_search_val_restricted becomes restricted dataset
#direct_search_small becomes smaller budget
#direct_search_large becomes Default Budget
#default_params becomes No Optimization
results_df["exp_name"] = results_df["exp_name"].replace({
    "val_vs_val_transformed":"Val. vs.\n Transformed",
    "mode2_id_vs_ood":"Max Prob vs.\n Transformed",
    "direct_search_val_restricted":"Smaller\n dataset",
    "direct_search_small":"Smaller\n budget",
    "direct_search_large":"Default\n budget",
    "default_params":"No opti-\nmization"
})

#ojbective paired_ood_acc becomes Paired Acc
results_df["objective"] = results_df["objective"].replace({
    "paired_ood_acc":"paired acc.",
    "auroc":"AUROC",
    "fpr95":"FPR95",
    "search":"Search",
    "none":"None"
})


In [ ]:
results_df


In [ ]:
from src.utils.eval.vis import plt_setup_latex

In [ ]:
W =plt_setup_latex()

In [ ]:

import os

def plot_per_run_results_transposed(results_df, objective_colors=None, figsize=(10, 8)):
    """
    Visualize per-run search accuracies for each experiment and objective (transposed),
    including standard error bars from 'accuracy_se'.
    """

    if objective_colors is None:
        objective_colors = {
            "AUROC": "#1f77b4",          # blue
            "FPR95": "#ff7f0e",          # orange
            "paired acc.": "#2ca02c",    # green
            "Search": "#d62728",         # red
            "None": "#7f7f7f",           # gray
        }

    experiments = results_df["exp_name"].unique()
    all_objectives = results_df["objective"].unique()
    n_runs = 6
    bar_height = 0.03
    fig, ax = plt.subplots(figsize=figsize)
    #print actual size
    print(f"Figure size: {fig.get_size_inches()} inches")

    current_y = 0
    y_labels_pos = []
    y_labels = []

    for exp_idx, exp in enumerate(experiments):
        exp_data = results_df[results_df["exp_name"] == exp]
        objectives_in_exp = exp_data["objective"].unique()
        n_objectives_in_exp = len(objectives_in_exp)
        exp_height = n_objectives_in_exp * n_runs * bar_height

        # Plot bars for each objective
        for obj_idx, obj in enumerate(objectives_in_exp):
            obj_data = exp_data[exp_data["objective"] == obj].sort_values("search_accuracy")
            color = objective_colors.get(obj, "#333333")
            accuracies = obj_data["search_accuracy"].values


            se_values = obj_data["search_accuracy_se"].values


            for run_idx in range(len(accuracies)):
                y_pos = current_y + (obj_idx * n_runs * bar_height) + (run_idx * bar_height)
                ax.barh(
                    y_pos,
                    accuracies[run_idx],
                    height=bar_height,
                    color=color,
                    alpha=0.7,
                    edgecolor='black',
                    linewidth=0.5,
                    xerr=se_values[run_idx],  # ← ADD ERROR BARS
                    error_kw=dict(ecolor='black', capsize=0.8, lw=0.5,
                                  capthick=0.5)
                )


        # Center label
        y_labels_pos.append(current_y + exp_height / 2 - bar_height / 2)
        y_labels.append(exp)
        current_y += exp_height + 0.04  # Add vertical gap
        #set fontsize 8 for y labels and ticks
    ax.tick_params(axis='y', labelsize=8)

    # Set y-axis labels
    ax.set_yticks(y_labels_pos)
    ax.set_yticklabels(y_labels, fontsize=8,linespacing=0.93)


    ax.set_xlabel("Accuracy", fontsize=9,labelpad=1)
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    ax.tick_params(axis='x', labelsize=8)
    ax.margins(y=0.01)

    # Legend
    #handles = [plt.Rectangle((0,0), 1, 1, color=objective_colors[obj], alpha=0.8)
    #for obj in all_objectives if obj in objective_colors]
    #ax.legend(handles, [obj for obj in all_objectives if obj in objective_colors],
    #          title="Objective", bbox_to_anchor=(1.02, 1), loc="upper
    #left", fontsize=8)

    # Set x-axis limits with padding
    x_min = results_df["search_accuracy"].min() - 0.012
    x_max = results_df["search_accuracy"].max() + 0.012
    ax.set_xlim(x_min, x_max)

    plt.tight_layout()
    path = os.path.join(current_path, "experiment_files", "export","fig", "results", "hyper_opt", dataset, transform_name)
    os.makedirs(path, exist_ok=True)
    plt.savefig(path + 'per_run_distributions_transposed.png', dpi=300, bbox_inches='tight')
    plt.savefig(path + 'per_run_distributions_transposed.pgf', bbox_inches='tight')
    plt.savefig(path + 'per_run_distributions_transposed.pdf', bbox_inches='tight')
    plt.show()
    print("Plot saved as 'per_run_distributions_transposed.png'")



plot_per_run_results_transposed(results_df, figsize=(W/2, W/2))


In [ ]:
from matplotlib.patches import Rectangle


def save_legend(objective_colors, all_objectives, base_path, filename="legend.png",
                title="Objective", fontsize=8, dpi=300):
    """
    Saves the legend generated from objective_colors and all_objectives to a separate file.
    """
    # Create the full saving path and ensure directory exists
    os.makedirs(base_path, exist_ok=True)
    full_filename = os.path.join(base_path, filename)

    legend_objectives = [obj for obj in all_objectives if obj in objective_colors]

    # Create handles (colored rectangles) for the legend
    handles = [
        Rectangle((0, 0), 1, 1, color=objective_colors[obj], alpha=0.8)
        for obj in legend_objectives
    ]
    labels = legend_objectives

    # Create a temporary figure and axes just for the legend
    fig_legend = plt.figure(figsize=(1.5, 1))
    ax_legend = fig_legend.add_subplot(111)

    # Generate the legend
    legend = ax_legend.legend(
        handles,
        labels,
        title=title,
        loc="center",
        fontsize=fontsize,
        frameon=False
    )
    legend.get_title().set_fontsize(fontsize)

    # Remove all other elements (axes, ticks, etc.)
    ax_legend.axis('off')
    ax_legend.set_frame_on(False)

    # Save the figure with only the legend
    try:
        # bbox_inches='tight' ensures the image is tightly cropped around the legend
        fig_legend.savefig(full_filename, dpi=dpi, bbox_inches='tight')
        print(f"\n Legend saved successfully to: {full_filename}")
    except Exception as e:
        print(f"\n❌ Error saving legend to {full_filename}: {e}")
    plt.tight_layout()
    plt.show()

In [ ]:
save_legend(
    objective_colors={
        "AUROC": "#1f77b4",          # blue
        "FPR95": "#ff7f0e",          # orange
        "Paired\n Accuracy": "#2ca02c",    # green
        "Search": "#d62728",         # red
        "None": "#7f7f7f",           # gray
    },
    all_objectives=["AUROC", "FPR95", "Paired\n Accuracy", "Search", "none"],
    base_path=os.path.join(current_path, "experiment_files", "export","fig", "results", "hyper_opt", dataset, transform_name),
    filename="legend.pdf",
    title="Objective",
    fontsize=9,
    dpi=300
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from matplotlib.patches import Rectangle

path = os.path.join(current_path, "experiment_files", "export","fig2", "results", "hyper_opt", dataset, transform_name)
os.makedirs(path, exist_ok=True)

def plot_per_run_results_transposed(results_df, objective_colors=None, figsize=(10, 8)):
    """
    Visualize per-run search accuracies for each experiment and objective (transposed),
    including standard error bars and a legend positioned outside the plot.
    """
    # Normalize objective names to match requested labels
    results_df = results_df.copy()
    results_df["objective"] = results_df["objective"].replace({
        "paired acc.": "Pairwise\n Accuracy",
        "None": "none"
    })

    if objective_colors is None:
        objective_colors = {
            "AUROC": "#1f77b4",          # blue
            "FPR95": "#ff7f0e",          # orange
            "Pairwise\n Accuracy": "#2ca02c", # green
            "Search": "#d62728",         # red
        }

    # Use specific experiments or all unique ones
    experiments = ["Default\n budget", "Val. vs.\n Transformed"]
    all_objectives = results_df["objective"].unique()

    n_runs = 6
    bar_height = 0.03
    fig, ax = plt.subplots(figsize=figsize)

    current_y = 0
    y_labels_pos = []
    y_labels = []

    for exp in experiments:
        exp_data = results_df[results_df["exp_name"] == exp]
        if exp_data.empty:
            continue

        objectives_in_exp = exp_data["objective"].unique()
        n_objectives_in_exp = len(objectives_in_exp)
        exp_height = n_objectives_in_exp * n_runs * bar_height

        for obj_idx, obj in enumerate(objectives_in_exp):
            obj_data = exp_data[exp_data["objective"] == obj].sort_values("search_accuracy")
            color = objective_colors.get(obj, "#333333")
            accuracies = obj_data["search_accuracy"].values
            se_values = obj_data["search_accuracy_se"].values

            for run_idx in range(len(accuracies)):
                y_pos = current_y + (obj_idx * n_runs * bar_height) + (run_idx * bar_height)
                ax.barh(
                    y_pos,
                    accuracies[run_idx],
                    height=bar_height,
                    color=color,
                    alpha=0.7,
                    edgecolor='black',
                    linewidth=0.5,
                    xerr=se_values[run_idx],
                    error_kw=dict(ecolor='black', capsize=0.8, lw=0.5, capthick=0.5)
                )

        y_labels_pos.append(current_y + exp_height / 2 - bar_height / 2)
        #y_labels.append(exp)
        current_y += exp_height + 0.04

    # Legend implementation - Positioned to the right of the plot
    legend_handles = [
        Rectangle((0, 0), 1, 1, color=objective_colors[obj], alpha=0.7)
        for obj in all_objectives if obj in objective_colors
    ]
    ax.legend(
        legend_handles,
        [obj for obj in all_objectives if obj in objective_colors],
        title="Objective",
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        fontsize=8
    )

    ax.tick_params(axis='y', labelsize=8)
    ax.set_yticks(y_labels_pos)
    ax.set_yticklabels(y_labels, fontsize=8, linespacing=0.93)
    ax.set_xlabel("Accuracy", fontsize=9, labelpad=1)
    ax.grid(axis="x", linestyle="--", alpha=0.4)
    ax.tick_params(axis='x', labelsize=8)

    x_min = results_df["search_accuracy"].min() - 0.012
    x_max = results_df["search_accuracy"].max() + 0.012
    ax.set_xlim(x_min, x_max)

    plt.tight_layout()
    plt.savefig(path + '/per_run_distributions_transposed_with_legend.pdf', bbox_inches='tight')
    plt.show()

W = plt_setup_paper()

plot_per_run_results_transposed(results_df, figsize=(W, W/3))


In [ ]:
results_df["exp_name"].unique()

In [ ]:
summary["max_accuracy"]